# **I. Canonical Data Preparation**

## **1. Build the Canonical Schema**

### **Design Decision: Retaining the Full 31-Column Schema**

We will retain **all 31 original columns** rather than stripping the dataset down to just the four required fields. 

**Rationale:** 
The core objective of this benchmark is to compare storage and I/O efficiency across different file formats using a realistic workload. Keeping the wider schema is particularly useful for the later Read Query Benchmark. The query accesses only `Carrier` and `ArrDelay` from a 31-column dataset, allowing columnar formats such as Parquet and ORC to benefit from physical column pruning.

CSV and JSON remain row-oriented/text-based formats and do not provide the same columnar storage-level skipping behavior.

We will only rename the four critical columns to standardize our canonical schema:

| Original Column | Canonical Column | Role |
|---|---|---|
| `YEAR` | `Year` | Partitioning |
| `MONTH` | `Month` | Partitioning |
| `AIRLINE` | `Carrier` | Grouping / Aggregation |
| `ARRIVAL_DELAY` | `ArrDelay` | Numeric Metric |

*Note: The remaining 27 columns retain their original names. No rows or columns are dropped during this initialization.*

In [1]:
# ============================================================
# BUILD THE CANONICAL DATAFRAME
# ============================================================

from pyspark.sql import SparkSession

INPUT_FILE = r"C:\BigDataProject\data\raw\airline\flights.csv"

# Start one SparkSession that will also be reused during benchmarking.
spark = (
    SparkSession.builder
    .appName("AirlineFileFormatBenchmark")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Read the complete raw dataset.
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_FILE)
)

# Keep all 31 columns and rename only the four fields required
# by the benchmark and partition experiments.
canonical_df = (
    raw_df
    .withColumnRenamed("YEAR", "Year")
    .withColumnRenamed("MONTH", "Month")
    .withColumnRenamed("AIRLINE", "Carrier")
    .withColumnRenamed("ARRIVAL_DELAY", "ArrDelay")
)

print(f"Number of columns: {len(canonical_df.columns)}")
print("\nCanonical columns:")
print(canonical_df.columns)

print("\nCanonical schema:")
canonical_df.printSchema()

Number of columns: 31

Canonical columns:
['Year', 'Month', 'DAY', 'DAY_OF_WEEK', 'Carrier', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ArrDelay', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

Canonical schema:
root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- Carrier: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIM

## **2. Missing-Value Policy for `ArrDelay`**

### **Decision: Preserve All Null Values**

The initial data audit identified 105,071 missing values in `ARRIVAL_DELAY`,
representing approximately 1.81% of the dataset. For our canonical benchmark dataset, these null values will be **strictly preserved**.

### **Rationale**

- **Data Scale Integrity:** Dropping rows (`dropna()`) would artificially shrink the benchmark dataset, undermining the volume needed for robust I/O testing.
- **Statistical Accuracy:** Imputing missing values (e.g., using `fillna(0)`) would alter the semantic meaning of the data and heavily bias the output of our target query `AVG(ArrDelay)`.
- **Spark Engine Behavior:** Spark SQL's `AVG()` aggregation natively ignores null values. Therefore, the Read Query Benchmark can execute accurately without requiring any imputation.
- **Storage Benchmarking:** Preserving nulls provides an excellent opportunity to observe how different file formats compress sparse data (e.g., Parquet/ORC's efficient null bitmasking).

**Final Policy Applied:**
- No `dropna()` or `fillna()` operations will be used.
- `ArrDelay` nulls are retained as-is (Expected null count: 105,071).

In [2]:
# ============================================================
# VALIDATE THE ARRIVAL DELAY NULL POLICY
# ============================================================

from pyspark.sql import functions as F

null_check = (
    canonical_df
    .agg(
        F.count("*").alias("total_rows"),
        F.sum(F.col("ArrDelay").isNull().cast("int")).alias("null_arrdelay")
    )
    .first()
)

total_rows = null_check["total_rows"]
null_arrdelay = null_check["null_arrdelay"]
null_percentage = null_arrdelay / total_rows * 100

print(f"Total rows       : {total_rows:,}")
print(f"Null ArrDelay    : {null_arrdelay:,}")
print(f"Null percentage  : {null_percentage:.4f}%")
print("\nPolicy: ArrDelay null values are preserved.")

Total rows       : 5,819,079
Null ArrDelay    : 105,071
Null percentage  : 1.8056%

Policy: ArrDelay null values are preserved.


## **3. Validate and Materialize the Canonical Benchmark Input**

Before executing the four file-format benchmarks, the canonical DataFrame must be rigorously validated and materialized.

### **3.1. Validation Constraints**
To ensure data integrity post-transformation, the input DataFrame must meet the exact parameters identified during the audit:
- **Dimensions:** Exactly 5,819,079 rows and 31 columns.
- **Partition Columns:** `Year` must strictly contain 2015; `Month` must cover values 1–12.
- **Grouping Column:** `Carrier` must contain 14 distinct airlines.
- **Numeric Column:** `ArrDelay` must retain its numeric data type and contain exactly 105,071 null values.

### **3.2. Materialization Strategy**
Once validated, the DataFrame will be cached using `persist(StorageLevel.MEMORY_AND_DISK)` and materialized via a `.count()` action.

**Rationale (Ensuring Benchmark Fairness):** 
Spark utilizes lazy evaluation. If we do not explicitly materialize the DataFrame beforehand, the upstream lineage (reading the raw CSV from disk and renaming columns) will be re-computed four separate times—once for each format write. 

By persisting and materializing the canonical DataFrame immediately before the benchmark runs, upstream CSV reading and schema-normalization transformations are not repeatedly recomputed for each format write.

This reduces upstream computation as a source of variation in Write Execution Time.

In [3]:
# ============================================================
# VALIDATE AND MATERIALIZE THE BENCHMARK INPUT
# ============================================================

from pyspark.storagelevel import StorageLevel
from pyspark.sql import functions as F

# Persist the canonical DataFrame so every format benchmark uses
# exactly the same materialized input.
benchmark_df = canonical_df.persist(StorageLevel.DISK_ONLY)

# Materialize the DataFrame once before any timed write operation.
materialized_row_count = benchmark_df.count()

# Validate the benchmark-critical fields.
validation = (
    benchmark_df
    .agg(
        F.min("Year").alias("min_year"),
        F.max("Year").alias("max_year"),
        F.countDistinct("Year").alias("distinct_years"),

        F.min("Month").alias("min_month"),
        F.max("Month").alias("max_month"),
        F.countDistinct("Month").alias("distinct_months"),

        F.countDistinct("Carrier").alias("distinct_carriers"),

        F.sum(F.col("ArrDelay").isNull().cast("int")).alias("null_arrdelay")
    )
    .first()
)

print(f"Rows               : {materialized_row_count:,}")
print(f"Columns            : {len(benchmark_df.columns)}")
print(f"Year range         : {validation['min_year']} - {validation['max_year']}")
print(f"Distinct years     : {validation['distinct_years']}")
print(f"Month range        : {validation['min_month']} - {validation['max_month']}")
print(f"Distinct months    : {validation['distinct_months']}")
print(f"Distinct carriers  : {validation['distinct_carriers']}")
print(f"Null ArrDelay      : {validation['null_arrdelay']:,}")

# Fail immediately if the canonical input is not what the benchmark expects.
assert materialized_row_count == 5_819_079
assert len(benchmark_df.columns) == 31
assert validation["min_year"] == 2015
assert validation["max_year"] == 2015
assert validation["distinct_years"] == 1
assert validation["min_month"] == 1
assert validation["max_month"] == 12
assert validation["distinct_months"] == 12
assert validation["distinct_carriers"] == 14
assert validation["null_arrdelay"] == 105_071
assert dict(benchmark_df.dtypes)["ArrDelay"] in ["int", "bigint", "float", "double"]

print("\nCanonical benchmark input: READY")

Rows               : 5,819,079
Columns            : 31
Year range         : 2015 - 2015
Distinct years     : 1
Month range        : 1 - 12
Distinct months    : 12
Distinct carriers  : 14
Null ArrDelay      : 105,071

Canonical benchmark input: READY


# **II. Benchmark Design**

## **1. Compression and Write Policy**

To evaluate the formats under standard operating conditions, this benchmark relies on **Spark's default compression settings** rather than forcing a uniform codec across all of them. 

Consequently, the final storage metrics represent a combination of **File Format + Default Compression**, rather than a strict format-only comparison.

### **Specific Format Policies:**
- **CSV & JSON:** Written as plain text without explicit compression. CSV will include a header.
- **Parquet & ORC:** Written using their active Spark defaults (typically Snappy in Spark 3.4.1). The exact codec will be dynamically read from the `SparkSession` and recorded in the benchmark metadata to ensure accuracy.

### **General Write Parameters:**
- **Write Mode:** `mode("overwrite")` is applied to all outputs to ensure clean, repeatable runs.
- **Partitioning:** No `.repartition()` or `.coalesce()` transformations are applied during this phase. (Partition control is evaluated in a separate experiment).

In [4]:
# ============================================================
# DEFINE COMPRESSION AND WRITE POLICY
# ============================================================

FORMATS = ["csv", "json", "parquet", "orc"]

# Record the actual compression configuration used by this SparkSession.
COMPRESSION_POLICY = {
    "csv": "default / no explicit compression option",
    "json": "default / no explicit compression option",
    "parquet": spark.conf.get("spark.sql.parquet.compression.codec"),
    "orc": spark.conf.get("spark.sql.orc.compression.codec")
}

# Define only format-specific options that are intentionally controlled.
WRITE_OPTIONS = {
    "csv": {"header": "true"},
    "json": {},
    "parquet": {},
    "orc": {}
}

print("Compression policy:")
for fmt in FORMATS:
    print(f"{fmt.upper():8} -> {COMPRESSION_POLICY[fmt]}")

Compression policy:
CSV      -> default / no explicit compression option
JSON     -> default / no explicit compression option
PARQUET  -> snappy
ORC      -> snappy


## **2. Output Directory Structure**

To maintain consistency and isolation, all benchmark outputs will be organized into a standardized directory structure:

```text
output/
└── format_benchmark/
    ├── csv/
    ├── json/
    ├── parquet/
    └── orc/
```

Each format is written to its designated path using `mode("overwrite")`. This approach ensures:

- **Data Isolation:** Outputs from different formats do not overlap or interfere with one another.
- **Reproducibility:** Previous results are cleanly replaced before each run, ensuring accurate disk size measurements without residual data.
- **Downstream Reliability:** The paths remain stable, providing fixed input locations for the subsequent Read Query experiments (to be executed by ID4).

*(Note: Future partition control experiments will be directed to a separate folder, such as `output/partition_test/`, to preserve the integrity of these baseline format benchmark results.)*

In [5]:
# ============================================================
# DEFINE BENCHMARK OUTPUT STRUCTURE
# ============================================================

import os

OUTPUT_ROOT = r"C:\BigDataProject\output\format_benchmark"

OUTPUT_PATHS = {
    fmt: os.path.join(OUTPUT_ROOT, fmt)
    for fmt in FORMATS
}

# Only the benchmark root is created now.
# Spark will create or overwrite each format directory during the write.
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Benchmark output paths:")
for fmt, path in OUTPUT_PATHS.items():
    print(f"{fmt.upper():8} -> {path}")

Benchmark output paths:
CSV      -> C:\BigDataProject\output\format_benchmark\csv
JSON     -> C:\BigDataProject\output\format_benchmark\json
PARQUET  -> C:\BigDataProject\output\format_benchmark\parquet
ORC      -> C:\BigDataProject\output\format_benchmark\orc


## **3. Benchmark Run Protocol**

To ensure consistent and reliable measurements, each file format will be evaluated using the following execution protocol:

### **Execution Steps**
1. **Warm-up Run (1 iteration):**
   - Executes the complete write operation to initialize the system (e.g., JVM and cache warm-up).
   - The execution time of this run is excluded from the final statistics.

2. **Measured Runs (3 iterations):**
   - Writes the materialized `benchmark_df` to the designated output path using `mode("overwrite")`.
   - Write Execution Time is recorded for each individual run.

3. **Reported Metric (Median):**
   - The final Write Execution Time will be reported as the median of the three measured runs. 
   - Using the median helps mitigate local execution variance caused by transient JVM activity, OS scheduling, or disk I/O fluctuations.

### **Control Variables**
To maintain a fair baseline comparison, all formats are tested under identical conditions:
- **Data:** Same input DataFrame (`benchmark_df`) and exact row count.
- **Environment:** Same `SparkSession`, Spark configuration, and host machine.
- **Partitioning:** Same number of input partitions (no repartitioning or coalescing is applied during this specific format benchmark).

Benchmark execution will be organized by rounds. Each warm-up or official round executes the formats in the fixed order CSV → JSON → Parquet → ORC.

In [6]:
# ============================================================
# DEFINE BENCHMARK RUN PROTOCOL
# ============================================================

WARMUP_RUNS = 1
OFFICIAL_RUNS = 3
SUMMARY_STATISTIC = "median"

# Record the partition count of the frozen canonical input.
# It must remain unchanged across all four format benchmarks.
INPUT_PARTITIONS = benchmark_df.rdd.getNumPartitions()

print(f"Warm-up runs      : {WARMUP_RUNS}")
print(f"Official runs     : {OFFICIAL_RUNS}")
print(f"Summary statistic : {SUMMARY_STATISTIC}")
print(f"Input rows        : {benchmark_df.count():,}")
print(f"Input partitions  : {INPUT_PARTITIONS}")

Warm-up runs      : 1
Official runs     : 3
Summary statistic : median
Input rows        : 5,819,079
Input partitions  : 16


## **4. Write Execution Time Boundaries**

Write Execution Time is measured using `time.perf_counter()`. The timer starts immediately before the Spark write action and stops immediately after `.save()`completes.

The measured interval therefore represents the end-to-end Spark write execution time for the already materialized input DataFrame. It includes format encoding, serialization, compression where applicable, task execution, and filesystem writing.

The following upstream operations are excluded from the timed interval:
- Reading the original CSV source.
- Canonical column renaming.
- Data validation.
- Initial persistence and materialization.
- Disk-size calculation.
- Result formatting.

This ensures that upstream data preparation is not repeatedly included in the vwrite benchmark.

In [7]:
# ============================================================
# DEFINE THE TIMED WRITE OPERATION
# ============================================================

import time

def timed_write(df, fmt, output_path):
    """Write one format once and return elapsed write execution time."""

    writer = df.write.mode("overwrite")

    # Apply only the predefined format-specific options.
    for key, value in WRITE_OPTIONS[fmt].items():
        writer = writer.option(key, value)

    # Start timing immediately before the Spark write action.
    start_time = time.perf_counter()

    writer.format(fmt).save(output_path)

    # Stop timing immediately after the write action completes.
    end_time = time.perf_counter()

    return end_time - start_time

## **5. Disk Storage Size Measurement**

To ensure a consistent comparison across CSV, JSON, Parquet, and ORC, the Disk Storage Size is calculated using a uniform methodology.

Because Spark writes dataset outputs as directories containing multiple distributed files rather than a single file, relying on standard directory metadata can yield inaccurate results. Therefore, the benchmark utilizes `os.walk()` to recursively scan the output directory and sum the sizes of the actual data files—specifically, those prefixed with `part-`.

**Exclusions:**
Non-data files such as `_SUCCESS` flags, CRC checksums, and other format-specific metadata are explicitly excluded from the calculation.

**Recorded Metrics:**
For each format, the benchmark will record:
- The total number of generated `part-*` files.
- The total disk storage size in bytes.
- The total disk storage size in megabytes (MB).

In [8]:
# ============================================================
# DEFINE DISK STORAGE SIZE MEASUREMENT
# ============================================================

def get_output_size(output_path):
    """Return the number and total size of Spark part-* data files."""

    total_bytes = 0
    part_file_count = 0

    for root, _, files in os.walk(output_path):
        for filename in files:
            if filename.startswith("part-"):
                file_path = os.path.join(root, filename)
                total_bytes += os.path.getsize(file_path)
                part_file_count += 1

    return {
        "part_file_count": part_file_count,
        "size_bytes": total_bytes,
        "size_mb": total_bytes / (1024 ** 2)
    }

## **6. Benchmark Result Structure**

Each official benchmark run generates a structured result record. This metadata ensures a seamless hand-off for downstream project phases, providing the necessary context for read query benchmarking, report compilation, and data visualization.

**Recorded Metrics per Run:**
- File format
- Active compression configuration
- Row count and input partition count
- Run iteration number
- Write Execution Time
- Output data-file count
- Disk Storage Size (in Bytes and MB)
- Output directory path

**Result Aggregation:**
- Warm-up executions are explicitly excluded from the final result table.
- The final benchmark summary reports the **median Write Execution Time** across the three measured runs to account for minor system variations.
- Because Disk Storage Size is deterministic for repeated writes of the same DataFrame, the size measured during the official runs will serve as the final storage metric.

In [9]:
# ============================================================
# DEFINE BENCHMARK RESULT STRUCTURE
# ============================================================

RESULT_COLUMNS = [
    "format",
    "compression",
    "row_count",
    "input_partitions",
    "run_number",
    "write_time_sec",
    "part_file_count",
    "size_bytes",
    "size_mb",
    "output_path"
]

# Official benchmark results will be appended here during Phase C.
benchmark_results = []

print("Official result fields:")
for column in RESULT_COLUMNS:
    print(f"- {column}")

Official result fields:
- format
- compression
- row_count
- input_partitions
- run_number
- write_time_sec
- part_file_count
- size_bytes
- size_mb
- output_path


# **III. Write Benchmark Execution**

## **1. Pre-run Integrity Check**

Before initiating the timed write operations, the benchmark environment is verified to ensure consistency. 

The objective of this check is to confirm the following conditions:
- The canonical `benchmark_df` is available and remains materialized/persisted.
- The row count and the number of input partitions remain constant.
- Output paths are properly configured for all four file formats.
- The established benchmark protocol remains active.

This step produces no benchmark measurements. It serves strictly as a validation gate to guarantee that all formats will be evaluated under identical input conditions before execution begins.

In [10]:
# ============================================================
# PRE-RUN INTEGRITY CHECK
# ============================================================

# Confirm that the canonical input is still cached and unchanged.
assert benchmark_df.is_cached, "benchmark_df is no longer cached."
assert benchmark_df.count() == materialized_row_count
assert benchmark_df.rdd.getNumPartitions() == INPUT_PARTITIONS

# Confirm that every benchmark format has a valid output path.
assert set(FORMATS) == set(OUTPUT_PATHS.keys())

print("Pre-run benchmark check: PASSED")
print(f"Rows             : {materialized_row_count:,}")
print(f"Columns          : {len(benchmark_df.columns)}")
print(f"Input partitions : {INPUT_PARTITIONS}")
print(f"Formats          : {', '.join(fmt.upper() for fmt in FORMATS)}")

Pre-run benchmark check: PASSED
Rows             : 5,819,079
Columns          : 31
Input partitions : 16
Formats          : CSV, JSON, PARQUET, ORC


## **2. Warm-up Round**

A complete warm-up iteration is executed prior to collecting official measurements. The formats are processed in a fixed sequence: **CSV → JSON → Parquet → ORC**.

**Purpose and Protocol:**
- This round replicates the exact write operations of the measured runs. Its primary purpose is to allow the Spark session and JVM to initialize, absorbing any first-run execution overhead.
- Execution times from the warm-up round are explicitly excluded from the final benchmark results.
- Since all write operations utilize `mode("overwrite")`, the data generated during the warm-up is safely replaced by the first official measured run, ensuring no residual data affects the downstream metrics.

In [11]:
# ============================================================
# EXECUTE ONE WARM-UP ROUND
# ============================================================

print("=" * 70)
print("WARM-UP ROUND")
print("=" * 70)

for fmt in FORMATS:
    print(f"Writing {fmt.upper()}...")

    # Use the exact same write function as the official benchmark,
    # but intentionally discard the measured time.
    timed_write(
        benchmark_df,
        fmt,
        OUTPUT_PATHS[fmt]
    )

    print(f"{fmt.upper()} warm-up completed.")

print("\nWarm-up round: COMPLETED")
print("No warm-up measurements will be included in the final results.")

WARM-UP ROUND
Writing CSV...
CSV warm-up completed.
Writing JSON...
JSON warm-up completed.
Writing PARQUET...
PARQUET warm-up completed.
Writing ORC...
ORC warm-up completed.

Warm-up round: COMPLETED
No warm-up measurements will be included in the final results.


## **3. Official Write Benchmark Runs**

Following the warm-up phase, three measured benchmark iterations are executed. To maintain consistency, the formats are processed in the same sequence during each round: **CSV → JSON → Parquet → ORC**.

**Execution Steps per Format:**
For every format in each iteration, the following steps are performed:
1. The materialized `benchmark_df` is written to disk using the `timed_write()` function.
2. The exact Write Execution Time is captured.
3. The output directory is scanned utilizing the `get_output_size()` function.
4. The total number of `part-*` data files and their aggregated disk size are recorded.
5. A structured result record is appended to the `benchmark_results` collection.

This procedure generates a total of 12 individual measurement records (4 formats × 3 iterations). 

*Note: To ensure metric accuracy, the directory scanning and size calculations are performed strictly after the timed write completes. This isolates the filesystem inspection overhead and prevents it from artificially inflating the Write Execution Time.*

In [12]:
# ============================================================
# EXECUTE THREE OFFICIAL BENCHMARK ROUNDS
# ============================================================

# Clear previous results so rerunning this cell does not create duplicates.
benchmark_results.clear()

for run_number in range(1, OFFICIAL_RUNS + 1):

    print("\n" + "=" * 70)
    print(f"OFFICIAL RUN {run_number}/{OFFICIAL_RUNS}")
    print("=" * 70)

    for fmt in FORMATS:

        output_path = OUTPUT_PATHS[fmt]

        print(f"\nWriting {fmt.upper()}...")

        # Measure the complete Spark write execution.
        write_time = timed_write(
            benchmark_df,
            fmt,
            output_path
        )

        # Measure output size only after the timed write has completed.
        size_info = get_output_size(output_path)

        result = {
            "format": fmt,
            "compression": COMPRESSION_POLICY[fmt],
            "row_count": materialized_row_count,
            "input_partitions": INPUT_PARTITIONS,
            "run_number": run_number,
            "write_time_sec": write_time,
            "part_file_count": size_info["part_file_count"],
            "size_bytes": size_info["size_bytes"],
            "size_mb": size_info["size_mb"],
            "output_path": output_path
        }

        benchmark_results.append(result)

        print(
            f"{fmt.upper():8} | "
            f"Time: {write_time:8.3f} s | "
            f"Size: {size_info['size_mb']:10.2f} MB | "
            f"Files: {size_info['part_file_count']}"
        )

print("\n" + "=" * 70)
print("OFFICIAL BENCHMARK EXECUTION COMPLETED")
print(f"Result records collected: {len(benchmark_results)}")
print("=" * 70)

assert len(benchmark_results) == len(FORMATS) * OFFICIAL_RUNS


OFFICIAL RUN 1/3

Writing CSV...
CSV      | Time:   11.307 s | Size:     562.36 MB | Files: 16

Writing JSON...
JSON     | Time:    3.524 s | Size:    2551.30 MB | Files: 16

Writing PARQUET...
PARQUET  | Time:    3.480 s | Size:     134.96 MB | Files: 16

Writing ORC...
ORC      | Time:    3.357 s | Size:     161.90 MB | Files: 16

OFFICIAL RUN 2/3

Writing CSV...
CSV      | Time:   10.825 s | Size:     562.36 MB | Files: 16

Writing JSON...
JSON     | Time:    3.414 s | Size:    2551.30 MB | Files: 16

Writing PARQUET...
PARQUET  | Time:    3.060 s | Size:     134.96 MB | Files: 16

Writing ORC...
ORC      | Time:    3.503 s | Size:     161.90 MB | Files: 16

OFFICIAL RUN 3/3

Writing CSV...
CSV      | Time:   10.901 s | Size:     562.36 MB | Files: 16

Writing JSON...
JSON     | Time:    3.071 s | Size:    2551.30 MB | Files: 16

Writing PARQUET...
PARQUET  | Time:    3.177 s | Size:     134.96 MB | Files: 16

Writing ORC...
ORC      | Time:    3.736 s | Size:     161.90 MB | Files

## **4. Raw Result Validation**

Prior to calculating the final aggregated metrics, the 12 raw measurement records undergo a validation step to ensure data consistency. 

**Validation Criteria:**
- Each format must have exactly three recorded iterations.
- Row counts and input partition counts must remain uniform across all runs.
- Every output directory must contain at least one `part-*` data file.
- Disk storage sizes must be successfully captured.

**Storage Consistency Check:**
Disk Storage Size is also cross-checked across the repeated runs. Given that the same DataFrame is written under identical configurations, the resulting storage footprint for a specific format is expected to remain stable. Any significant variance across iterations would prompt a review of the benchmark environment before the final summary is produced.

In [13]:
# ============================================================
# VALIDATE AND DISPLAY RAW BENCHMARK RESULTS
# ============================================================

results_df = (
    spark.createDataFrame(benchmark_results)
    .select(*RESULT_COLUMNS)
    .orderBy("run_number", "format")
)

results_df.show(20, truncate=False)

# Validate each format independently.
for fmt in FORMATS:

    format_results = [
        result
        for result in benchmark_results
        if result["format"] == fmt
    ]

    assert len(format_results) == OFFICIAL_RUNS
    assert all(r["row_count"] == materialized_row_count for r in format_results)
    assert all(r["input_partitions"] == INPUT_PARTITIONS for r in format_results)
    assert all(r["part_file_count"] > 0 for r in format_results)
    assert all(r["size_bytes"] > 0 for r in format_results)

    sizes = [r["size_bytes"] for r in format_results]

    print(
        f"{fmt.upper():8} | "
        f"Runs: {len(format_results)} | "
        f"Size consistent: {len(set(sizes)) == 1}"
    )

print("\nRaw benchmark validation: PASSED")

+-------+----------------------------------------+---------+----------------+----------+------------------+---------------+----------+------------------+-------------------------------------------------+
|format |compression                             |row_count|input_partitions|run_number|write_time_sec    |part_file_count|size_bytes|size_mb           |output_path                                      |
+-------+----------------------------------------+---------+----------------+----------+------------------+---------------+----------+------------------+-------------------------------------------------+
|csv    |default / no explicit compression option|5819079  |16              |1         |11.30700859999979 |16             |589677593 |562.3603754043579 |C:\BigDataProject\output\format_benchmark\csv    |
|json   |default / no explicit compression option|5819079  |16              |1         |3.524053300000105 |16             |2675232631|2551.3006505966187|C:\BigDataProject\output\format

## **5. Final Benchmark Summary**

The final Write Execution Time for each file format is reported as the **median of the three measured runs**. The data generated during the final iteration (Run 3) is preserved to serve as the standard input for subsequent Read Query Benchmarks.

**Aggregated Metrics:**
For each format, the final summary compiles the following details:
- Active compression configuration
- Number of measured iterations
- Median Write Execution Time
- Final `part-*` data file count
- Final Disk Storage Size
- Target output directory path

**Result Export and Storage:**
Both the detailed 12-run dataset and the aggregated four-format summary are exported as CSV metadata files. To ensure these files do not inflate the `part-*` disk size measurements, they are saved in the benchmark's root directory rather than within the format-specific output folders.

In [14]:
# ============================================================
# CALCULATE FINAL MEDIANS AND SAVE BENCHMARK RESULTS
# ============================================================

import csv
import statistics

benchmark_summary = []

for fmt in FORMATS:

    format_results = [
        result
        for result in benchmark_results
        if result["format"] == fmt
    ]

    write_times = [
        result["write_time_sec"]
        for result in format_results
    ]

    # The current directory contains the output from Official Run 3.
    final_run = max(
        format_results,
        key=lambda result: result["run_number"]
    )

    benchmark_summary.append({
        "format": fmt,
        "compression": COMPRESSION_POLICY[fmt],
        "official_runs": OFFICIAL_RUNS,
        "median_write_time_sec": statistics.median(write_times),
        "part_file_count": final_run["part_file_count"],
        "size_bytes": final_run["size_bytes"],
        "size_mb": final_run["size_mb"],
        "output_path": final_run["output_path"]
    })


# Display the canonical four-format summary.
summary_df = spark.createDataFrame(benchmark_summary)

summary_df.select(
    "format",
    "compression",
    "official_runs",
    "median_write_time_sec",
    "part_file_count",
    "size_mb"
).orderBy("format").show(truncate=False)


# Save the detailed 12-run results.
detailed_results_path = os.path.join(
    OUTPUT_ROOT,
    "write_benchmark_runs.csv"
)

with open(detailed_results_path, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=RESULT_COLUMNS
    )

    writer.writeheader()
    writer.writerows(benchmark_results)


# Save the final four-format summary.
summary_path = os.path.join(
    OUTPUT_ROOT,
    "write_benchmark_summary.csv"
)

summary_columns = [
    "format",
    "compression",
    "official_runs",
    "median_write_time_sec",
    "part_file_count",
    "size_bytes",
    "size_mb",
    "output_path"
]

with open(summary_path, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=summary_columns
    )

    writer.writeheader()
    writer.writerows(benchmark_summary)


print("\n" + "=" * 70)
print("PHASE C COMPLETED")
print("=" * 70)

print(f"Detailed results : {detailed_results_path}")
print(f"Final summary    : {summary_path}")

print("\nCanonical outputs for ID4:")

for fmt in FORMATS:
    print(f"{fmt.upper():8} -> {OUTPUT_PATHS[fmt]}")

+-------+----------------------------------------+-------------+---------------------+---------------+------------------+
|format |compression                             |official_runs|median_write_time_sec|part_file_count|size_mb           |
+-------+----------------------------------------+-------------+---------------------+---------------+------------------+
|csv    |default / no explicit compression option|3            |10.900860799999919   |16             |562.3603754043579 |
|json   |default / no explicit compression option|3            |3.4135555999998815   |16             |2551.3006505966187|
|orc    |snappy                                  |3            |3.5026201000000583   |16             |161.9030361175537 |
|parquet|snappy                                  |3            |3.1768253000000186   |16             |134.9634494781494 |
+-------+----------------------------------------+-------------+---------------------+---------------+------------------+


PHASE C COMPLETED
Deta

# **IV. Handoff to ID4**

## **1. Benchmark Result Finalization**

Prior to transitioning the outputs for subsequent read query and partition experiments, the benchmark result set is verified and finalized.

**Handoff Deliverables:**
- The four output directories generated during the final measured run.
- The comprehensive 12-run benchmark result dataset.
- The aggregated median summary for all four formats.
- The standardized 31-column dataset schema.
- The benchmark protocol established in the preceding phases.

**Final Consistency Verification:**
A concluding check is performed to ensure the integrity of the deliverables:
- All four format output directories are present and contain valid `part-*` data files.
- Exactly 12 measurement records and 4 summary records have been successfully compiled.
- The canonical row count and input partition count remain uniform across all format outputs.

*Note: This step serves strictly as a verification checkpoint; no benchmark operations are re-executed.*

In [15]:
# ============================================================
# FREEZE AND VALIDATE FINAL BENCHMARK RESULTS
# ============================================================

EXPECTED_RESULT_COUNT = len(FORMATS) * OFFICIAL_RUNS

# Validate the raw benchmark result set.
assert len(benchmark_results) == EXPECTED_RESULT_COUNT
assert len(benchmark_summary) == len(FORMATS)

# Validate the canonical output directories.
for fmt in FORMATS:
    output_path = OUTPUT_PATHS[fmt]
    size_info = get_output_size(output_path)

    assert os.path.isdir(output_path), f"Missing output directory: {output_path}"
    assert size_info["part_file_count"] > 0, f"No part-* files found for {fmt}"
    assert size_info["size_bytes"] > 0, f"Empty output detected for {fmt}"

# Validate benchmark conditions across all official runs.
assert all(
    result["row_count"] == materialized_row_count
    for result in benchmark_results
)

assert all(
    result["input_partitions"] == INPUT_PARTITIONS
    for result in benchmark_results
)

print("=" * 70)
print("FINAL BENCHMARK FREEZE CHECK")
print("=" * 70)
print(f"Official result records : {len(benchmark_results)}")
print(f"Final format summaries  : {len(benchmark_summary)}")
print(f"Canonical rows          : {materialized_row_count:,}")
print(f"Input partitions        : {INPUT_PARTITIONS}")

for fmt in FORMATS:
    size_info = get_output_size(OUTPUT_PATHS[fmt])

    print(
        f"{fmt.upper():8} | "
        f"Files: {size_info['part_file_count']:>3} | "
        f"Size: {size_info['size_mb']:>10.2f} MB"
    )

print("\nBenchmark result set: FROZEN")

FINAL BENCHMARK FREEZE CHECK
Official result records : 12
Final format summaries  : 4
Canonical rows          : 5,819,079
Input partitions        : 16
CSV      | Files:  16 | Size:     562.36 MB
JSON     | Files:  16 | Size:    2551.30 MB
PARQUET  | Files:  16 | Size:     134.96 MB
ORC      | Files:  16 | Size:     161.90 MB

Benchmark result set: FROZEN


## **2. Export Canonical Schema and Benchmark Protocol**

To maintain consistency during the subsequent Read Query Time and Partition experiments, the handoff package explicitly documents the dataset specifications and benchmarking protocol.

### **Canonical Dataset Specifications**
- **Dataset:** 2015 Flight Delays and Cancellations (`flights.csv`)
- **Dimensions:** 5,819,079 rows × 31 columns
- **Preprocessing Policy:** No rows dropped; no imputation performed (null values in `ArrDelay` are preserved).

### **Canonical Column Mapping**
- `YEAR` → `Year`
- `MONTH` → `Month`
- `AIRLINE` → `Carrier`
- `ARRIVAL_DELAY` → `ArrDelay`

### **Benchmark Protocol Baseline**
- **Input Data:** A single, materialized DataFrame utilizing the `DISK_ONLY` persistence strategy.
- **Environment:** Uniform `SparkSession` and runtime configuration.
- **Execution Sequence:** CSV → JSON → Parquet → ORC.
- **Iteration Structure:** 1 warm-up round followed by 3 measured runs.
- **Metrics Calculation:** 
  - *Write Execution Time:* Median of the 3 measured runs.
  - *Disk Storage Size:* Aggregated size of `part-*` data files exclusively.
- **Write Configurations:** 
  - All outputs use `mode("overwrite")`.
  - CSV output includes a header.
  - Parquet and ORC compression settings are dynamically recorded from the active Spark runtime.

These documented parameters establish the controlled baseline required for the downstream Read Query Benchmark.

In [16]:
# ============================================================
# EXPORT CANONICAL SCHEMA AND BENCHMARK PROTOCOL
# ============================================================

import json

HANDOFF_DIR = r"C:\BigDataProject\output\id4_handoff"
os.makedirs(HANDOFF_DIR, exist_ok=True)

# Export the canonical Spark schema.
schema_path = os.path.join(
    HANDOFF_DIR,
    "canonical_schema.json"
)

with open(schema_path, "w", encoding="utf-8") as file:
    json.dump(
        json.loads(benchmark_df.schema.json()),
        file,
        indent=4
    )

# Export the benchmark protocol.
protocol = {
    "dataset": {
        "name": "2015 Flight Delays and Cancellations",
        "source_file": INPUT_FILE,
        "row_count": materialized_row_count,
        "column_count": len(benchmark_df.columns),
        "arrdelay_null_policy": "preserve",
        "rows_removed": 0
    },

    "canonical_mapping": {
        "YEAR": "Year",
        "MONTH": "Month",
        "AIRLINE": "Carrier",
        "ARRIVAL_DELAY": "ArrDelay"
    },

    "benchmark": {
        "formats": FORMATS,
        "format_order": FORMATS,
        "warmup_runs": WARMUP_RUNS,
        "official_runs": OFFICIAL_RUNS,
        "summary_statistic": SUMMARY_STATISTIC,
        "input_partitions": INPUT_PARTITIONS,
        "persistence": "DISK_ONLY",
        "compression_policy": COMPRESSION_POLICY,
        "write_mode": "overwrite",
        "csv_header": True,
        "disk_size_rule": "recursive sum of part-* files only"
    },

    "spark": {
        "spark_version": spark.version,
        "master": spark.sparkContext.master,
        "shuffle_partitions": spark.conf.get(
            "spark.sql.shuffle.partitions"
        )
    }
}

protocol_path = os.path.join(
    HANDOFF_DIR,
    "benchmark_protocol.json"
)

with open(protocol_path, "w", encoding="utf-8") as file:
    json.dump(
        protocol,
        file,
        indent=4
    )

print(f"Schema exported   : {schema_path}")
print(f"Protocol exported : {protocol_path}")

Schema exported   : C:\BigDataProject\output\id4_handoff\canonical_schema.json
Protocol exported : C:\BigDataProject\output\id4_handoff\benchmark_protocol.json


## **3. Result Packaging and Handoff**

To optimize disk storage and prevent unnecessary data duplication, the four generated datasets are retained in their original benchmark directories rather than being copied into a separate handoff folder:

- `output/format_benchmark/csv/`
- `output/format_benchmark/json/`
- `output/format_benchmark/parquet/`
- `output/format_benchmark/orc/`

**Handoff Manifest:**
The handoff package includes an output manifest that records the canonical path, active compression configuration, data file count, disk storage size, and median Write Execution Time for each format. 

Additionally, the detailed 12-run measurement table and the final aggregated summary are provided within the package. The directory paths specified in this manifest will serve as the direct inputs for the downstream Read Query Time benchmark.

In [17]:
# ============================================================
# PACKAGE BENCHMARK RESULTS AND OUTPUT MANIFEST
# ============================================================

import csv
import shutil

# Copy the benchmark result tables into the handoff directory.
shutil.copy2(
    detailed_results_path,
    os.path.join(
        HANDOFF_DIR,
        "write_benchmark_runs.csv"
    )
)

shutil.copy2(
    summary_path,
    os.path.join(
        HANDOFF_DIR,
        "write_benchmark_summary.csv"
    )
)

# Build one canonical manifest entry for each format.
output_manifest = []

for summary in benchmark_summary:
    output_manifest.append({
        "format": summary["format"],
        "compression": summary["compression"],
        "row_count": materialized_row_count,
        "input_partitions": INPUT_PARTITIONS,
        "part_file_count": summary["part_file_count"],
        "size_bytes": summary["size_bytes"],
        "size_mb": summary["size_mb"],
        "median_write_time_sec": summary["median_write_time_sec"],
        "output_path": summary["output_path"]
    })

manifest_path = os.path.join(
    HANDOFF_DIR,
    "output_manifest.csv"
)

with open(
    manifest_path,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=output_manifest[0].keys()
    )

    writer.writeheader()
    writer.writerows(output_manifest)

print("=" * 70)
print("ID4 OUTPUT MANIFEST")
print("=" * 70)

for item in output_manifest:
    print(
        f"{item['format'].upper():8} | "
        f"{item['size_mb']:>10.2f} MB | "
        f"Median Write: {item['median_write_time_sec']:>8.3f} s | "
        f"{item['output_path']}"
    )

print(f"\nManifest saved: {manifest_path}")

ID4 OUTPUT MANIFEST
CSV      |     562.36 MB | Median Write:   10.901 s | C:\BigDataProject\output\format_benchmark\csv
JSON     |    2551.30 MB | Median Write:    3.414 s | C:\BigDataProject\output\format_benchmark\json
PARQUET  |     134.96 MB | Median Write:    3.177 s | C:\BigDataProject\output\format_benchmark\parquet
ORC      |     161.90 MB | Median Write:    3.503 s | C:\BigDataProject\output\format_benchmark\orc

Manifest saved: C:\BigDataProject\output\id4_handoff\output_manifest.csv


## **4. Final Handoff Package**

The final handoff package consolidates all data outputs and metadata necessary to execute the second half of Task 2.

**Dataset Outputs:**
- CSV output directory
- JSON output directory
- Parquet output directory
- ORC output directory

**Benchmark Metadata:**
- `canonical_schema.json`
- `benchmark_protocol.json`
- `output_manifest.csv`
- `write_benchmark_runs.csv`
- `write_benchmark_summary.csv`

### **Downstream Tasks**
Using these standardized outputs, the subsequent phase will execute the following operations:
1. Read Query Time comparison.
2. `.repartition(20)` experiment.
3. `.coalesce(2)` experiment.
4. `.partitionBy("Year", "Month")` experiment.

*Note: To maintain experimental integrity, the dataset, schema, compression policy, and benchmark assumptions established in this phase should remain unaltered unless explicitly coordinated across tasks.*

Upon successful generation of the handoff package, the persisted benchmark DataFrame is unpersisted from memory/disk, and the SparkSession is safely terminated.

In [18]:
# ============================================================
# CREATE HANDOFF README AND CLOSE THE SPARK SESSION
# ============================================================

readme_path = os.path.join(
    HANDOFF_DIR,
    "README_ID4.txt"
)

readme_text = f"""
ID4 HANDOFF — TASK 2 FILE FORMAT BENCHMARK
==========================================

DATASET
-------
2015 Flight Delays and Cancellations
Rows: {materialized_row_count:,}
Columns: {len(benchmark_df.columns)}

CANONICAL FIELDS
----------------
YEAR          -> Year
MONTH         -> Month
AIRLINE       -> Carrier
ARRIVAL_DELAY -> ArrDelay

NULL POLICY
-----------
ArrDelay null values are preserved.
No rows were dropped or imputed.

CANONICAL OUTPUTS
-----------------
CSV:     {OUTPUT_PATHS["csv"]}
JSON:    {OUTPUT_PATHS["json"]}
PARQUET: {OUTPUT_PATHS["parquet"]}
ORC:     {OUTPUT_PATHS["orc"]}

WRITE BENCHMARK PROTOCOL
------------------------
Warm-up runs: {WARMUP_RUNS}
Official runs: {OFFICIAL_RUNS}
Final timing statistic: {SUMMARY_STATISTIC}
Input partitions: {INPUT_PARTITIONS}
Persistence strategy: DISK_ONLY
Format order: CSV -> JSON -> PARQUET -> ORC

NEXT TASKS FOR ID4
------------------
1. Benchmark Read Query Time across all four formats.
2. Run repartition(20) experiment.
3. Run coalesce(2) experiment.
4. Run partitionBy("Year", "Month") experiment.
5. Return read/partition results to ID3 for the final canonical benchmark set.

IMPORTANT
---------
Do not change the dataset, schema, benchmark paths, or protocol without
coordinating with ID3.
"""

with open(
    readme_path,
    "w",
    encoding="utf-8"
) as file:
    file.write(readme_text.strip())

print("=" * 70)
print("ID4 HANDOFF PACKAGE READY")
print("=" * 70)

for filename in sorted(os.listdir(HANDOFF_DIR)):
    print(f"- {filename}")

print(f"\nHandoff directory: {HANDOFF_DIR}")

# Release the persisted canonical input only after all benchmark work is finished.
benchmark_df.unpersist()

# Stop Spark cleanly at the very end of the notebook.
spark.stop()

print("\nbenchmark_df released.")
print("SparkSession stopped.")
print("ID3 WRITE BENCHMARK WORKFLOW: COMPLETED")

ID4 HANDOFF PACKAGE READY
- README_ID4.txt
- benchmark_protocol.json
- canonical_schema.json
- output_manifest.csv
- write_benchmark_runs.csv
- write_benchmark_summary.csv

Handoff directory: C:\BigDataProject\output\id4_handoff

benchmark_df released.
SparkSession stopped.
ID3 WRITE BENCHMARK WORKFLOW: COMPLETED
